# Part 1b: can the bracket predict forgetting WITHOUT the old data?

After learning task A, we keep only a small **stored summary** of A and never touch A's data again:
- `g_A`: A's gradient at the end of training (one vector)
- the **top-K curvature directions** of A (Hessian eigenvectors + eigenvalues), the "map" of where A is fragile
- EWC's importance numbers (**diagonal Fisher**), so EWC gets its own fair stored summary

While training on B, three **no-old-data** signals are measured over the first 20 steps:
1. **Bracket (stored):** H_B·ĝ_A − (stored H_A)·g_B
2. **EWC penalty:** Σ F_i (θ_i − θ_A,i)², exactly the quantity EWC uses
3. **Estimated watching:** A's loss rise predicted from the stored map (g_A·Δθ + ½ΔθᵀHΔθ)

For reference only (these DO use A's data): the full bracket and the real simple check.

Run cells top to bottom. Use the same kernel as before (`E:\wordstest\.venv\Scripts\python.exe`).

## Cell 1: settings
**Should print:** the folders.

In [1]:
import os
BASE = r"F:\forgetting"
DATA_DIR = os.path.join(BASE, "data")
RESULTS_DIR = os.path.join(BASE, "results_stored")
os.makedirs(RESULTS_DIR, exist_ok=True)
SEEDS = 5
EPOCHS = 2
LR = 0.05
BATCH = 64
HIDDEN = 256
EST_SIZE = 2000
K = 20                 # number of curvature directions kept in the stored map
print("data:", DATA_DIR); print("results:", RESULTS_DIR)

data: F:\forgetting\data
results: F:\forgetting\results_stored


## Cell 2: GPU + libraries
**Should print:** `cuda` and the 3090.

In [2]:
import time, json, csv
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.func import functional_call, vmap, grad as fgrad
from scipy.stats import spearmanr, rankdata
from scipy.sparse.linalg import LinearOperator, eigsh
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", dev)
if dev.type == "cuda": print(torch.cuda.get_device_name(0))

torch 2.5.1+cu121 | device: cuda
NVIDIA GeForce RTX 3090


## Cell 3: load MNIST (already on your drive, no download)
**Should print:** `train 60000 test 10000` and the five task lines.

In [3]:
from torchvision import datasets
tr = datasets.MNIST(DATA_DIR, train=True, download=False)
te = datasets.MNIST(DATA_DIR, train=False, download=False)
def prep(ds):
    X = ds.data.float().div(255.).view(-1, 784)
    return (X - 0.1307) / 0.3081, ds.targets.clone()
Xtr_all, ytr_all = prep(tr); Xte_all, yte_all = prep(te)
print("train", len(ytr_all), "test", len(yte_all))
TASKS = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]
NAMES = ["0 vs 1", "2 vs 3", "4 vs 5", "6 vs 7", "8 vs 9"]
data = {}
g0 = torch.Generator().manual_seed(123)
for t, cls in enumerate(TASKS):
    mtr = (ytr_all == cls[0]) | (ytr_all == cls[1]); mte = (yte_all == cls[0]) | (yte_all == cls[1])
    Xtr, ytr, Xte, yte = Xtr_all[mtr], ytr_all[mtr], Xte_all[mte], yte_all[mte]
    est = torch.randperm(len(ytr), generator=g0)[:EST_SIZE]
    data[t] = dict(Xtr=Xtr.to(dev), ytr=ytr.to(dev), Xte=Xte.to(dev), yte=yte.to(dev),
                   Xest=Xtr[est].to(dev), yest=ytr[est].to(dev), cls=torch.tensor(cls, device=dev))
    print(f"task {t} ({NAMES[t]}): train {len(ytr)}, test {len(yte)}")

train 60000 test 10000
task 0 (0 vs 1): train 12665, test 2115
task 1 (2 vs 3): train 12089, test 2042
task 2 (4 vs 5): train 11263, test 1874
task 3 (6 vs 7): train 12183, test 1986
task 4 (8 vs 9): train 11800, test 1983


## Cell 4: model and maths (same as before)
**Should print:** `ready`.

In [4]:
SCENARIO = "task"
class MLP(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.l1 = nn.Linear(784, HIDDEN); self.l2 = nn.Linear(HIDDEN, HIDDEN); self.l3 = nn.Linear(HIDDEN, out_dim)
    def forward(self, x):
        return self.l3(torch.tanh(self.l2(torch.tanh(self.l1(x)))))

def setup(scenario):
    global SCENARIO, model, P_NAMES, P_SHAPES, P_SIZES, NPARAM
    SCENARIO = scenario
    model = MLP(2 if scenario == "domain" else 10).to(dev)
    P_NAMES = [n for n, _ in model.named_parameters()]
    P_SHAPES = [p.shape for _, p in model.named_parameters()]
    P_SIZES = [p.numel() for _, p in model.named_parameters()]
    NPARAM = sum(P_SIZES)

def unflat(th):
    out, i = {}, 0
    for n, s, k in zip(P_NAMES, P_SHAPES, P_SIZES):
        out[n] = th[i:i + k].view(s); i += k
    return out

def init(seed):
    torch.manual_seed(seed)
    m = MLP(2 if SCENARIO == "domain" else 10).to(dev)
    return torch.cat([p.detach().flatten() for p in m.parameters()])

def logits(th, X, cls):
    z = functional_call(model, unflat(th), (X,))
    return z if SCENARIO == "domain" else z[:, cls]

def task_loss(th, X, y, cls):
    return F.cross_entropy(logits(th, X, cls), (y == cls[1]).long())

def grad(th, t, X=None, y=None):
    d = data[t]; X = d["Xest"] if X is None else X; y = d["yest"] if y is None else y
    th = th.detach().requires_grad_(True)
    return torch.autograd.grad(task_loss(th, X, y, d["cls"]), th)[0].detach()

def hvp(th, t, v):
    d = data[t]; th = th.detach().requires_grad_(True)
    g = torch.autograd.grad(task_loss(th, d["Xest"], d["yest"], d["cls"]), th, create_graph=True)[0]
    return torch.autograd.grad(g @ v.detach(), th)[0].detach()

def bracket(th, a, b):
    ga, gb = grad(th, a), grad(th, b)
    return hvp(th, b, ga) - hvp(th, a, gb)

@torch.no_grad()
def evaluate(th, t, split="te"):
    d = data[t]; X, y = d["X" + split], d["y" + split]
    z = logits(th, X, d["cls"]); yi = (y == d["cls"][1]).long()
    return float((z.argmax(1) == yi).float().mean()), float(F.cross_entropy(z, yi))

def train(th, t, seed, record=None, record_every=5):
    d = data[t]; th = th.detach().clone()
    gen = torch.Generator().manual_seed(seed * 1000 + t)
    n = len(d["ytr"]); step = 0
    for ep in range(EPOCHS):
        idx = torch.randperm(n, generator=gen).to(dev)
        for i in range(0, n, BATCH):
            bi = idx[i:i + BATCH]
            if record is not None and step % record_every == 0: record(step, th)
            th = th - LR * grad(th, t, d["Xtr"][bi], d["ytr"][bi])
            step += 1
    if record is not None: record(step, th)
    return th
print("ready")

ready


## Cell 5: the stored summary
After task A is learned, this builds A's small summary. **After this, A's data is never used by the no-data signals.**
- top-K Hessian directions (Lanczos), so the stored map is K vectors
- diagonal Fisher for EWC (average squared per-image gradients)

**Should print:** `defined`.

In [5]:
def top_hessian(th, t, k=K):
    # top-k eigenvalues/eigenvectors of task t's Hessian at th, using only Hessian-vector products
    def mv(v):
        v_t = torch.as_tensor(np.asarray(v).ravel(), dtype=torch.float32, device=dev)
        return hvp(th, t, v_t).double().cpu().numpy()
    op = LinearOperator((NPARAM, NPARAM), matvec=mv, dtype=np.float64)
    vals, vecs = eigsh(op, k=k, which="LA", tol=1e-3)
    return torch.tensor(vals, dtype=torch.float32, device=dev), torch.tensor(vecs, dtype=torch.float32, device=dev)

def diag_fisher(th, t, chunk=250):
    d = data[t]; cls = d["cls"]
    def one(th_, x, y):
        return task_loss(th_, x[None], y[None], cls)
    per = vmap(fgrad(one), in_dims=(None, 0, 0))
    Fd = torch.zeros(NPARAM, device=dev)
    for i in range(0, len(d["yest"]), chunk):
        g = per(th.detach(), d["Xest"][i:i + chunk], d["yest"][i:i + chunk])
        Fd += (g ** 2).sum(0)
    return Fd / len(d["yest"])

def make_summary(th, t):
    lam, U = top_hessian(th, t)
    return dict(theta=th.detach().clone(), g=grad(th, t), lam=lam, U=U, F=diag_fisher(th, t))

def H_stored(S, v):
    # stored low-rank Hessian of the old task times v
    return S["U"] @ (S["lam"] * (S["U"].T @ v))

def stored_signals(S, th, b):
    # all computed WITHOUT the old task's data (only its summary S, plus task B's data)
    dth = th - S["theta"]
    gA_hat = S["g"] + H_stored(S, dth)                 # old task's gradient, predicted from the map
    gB = grad(th, b)
    br = hvp(th, b, gA_hat) - H_stored(S, gB)          # bracket from the stored map
    ewc = float((S["F"] * dth ** 2).sum())            # EWC penalty
    Ud = S["U"].T @ dth
    est = float(S["g"] @ dth + 0.5 * (S["lam"] * Ud ** 2).sum())   # estimated loss rise of old task
    return float(br.norm()), ewc, est, gB
print("defined")

defined


## Cell 6: sanity check (about a minute)
Compares the stored-map bracket with the full bracket (which uses A's data) for one pair.

**Should print:** top eigenvalues (positive, falling), then a direction match and size ratio at the switch and after 20 steps. A direction match above about 0.5 means the stored map captures the bracket reasonably well. Send this output to Claude.

In [ ]:
setup("domain")
t0 = time.time()
thA = train(init(0), 2, 0)
S = make_summary(thA, 2)
print(f"summary built in {time.time() - t0:.1f}s | top eigenvalues:", np.round(S["lam"].flip(0)[:6].cpu().numpy(), 3))
def compare(th):
    full = bracket(th, 2, 4)
    dth = th - S["theta"]; gA_hat = S["g"] + H_stored(S, dth); gB = grad(th, 4)
    st = hvp(th, 4, gA_hat) - H_stored(S, gB)
    return float(F.cosine_similarity(full, st, dim=0)), float(st.norm() / full.norm())
print("at the switch:   direction match %.3f, size ratio %.3f" % compare(thA))
th20 = thA.clone()
d = data[4]; gen = torch.Generator().manual_seed(7); idx = torch.randperm(len(d["ytr"]), generator=gen).to(dev)
for i in range(20):
    bi = idx[i * BATCH:(i + 1) * BATCH]; th20 = th20 - LR * grad(th20, 4, d["Xtr"][bi], d["ytr"][bi])
print("after 20 steps:  direction match %.3f, size ratio %.3f" % compare(th20))

## Cell 7: the experiment function
**Should print:** `defined`.

In [ ]:
SIGNALS = {
    "bracket_stored": "Bracket from stored map (no old data)",
    "ewc_penalty": "EWC penalty (no old data)",
    "est_watch": "Estimated watching from stored map (no old data)",
    "curv_stored_switch": "EWC-style curvature at switch (no old data)",
    "bracket_full": "Reference: full bracket (uses old data)",
    "simple_check": "Reference: simple check (uses old data)",
}

def run(scenario, lr=0.05, seeds=SEEDS, tag=None):
    global LR
    LR = lr
    setup(scenario)
    tag = tag or f"{scenario}_lr{lr}"
    out = os.path.join(RESULTS_DIR, tag); os.makedirs(out, exist_ok=True)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    rows, t0 = [], time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base = {t: evaluate(th_task[t], t) for t in range(5)}
        base_tr = {t: evaluate(th_task[t], t, "est")[1] for t in range(5)}
        summ = {t: make_summary(th_task[t], t) for t in range(5)}
        for a, b in pairs:
            S = summ[a]; log = []
            def rec(step, th_, a=a, b=b, S=S, log=log):
                if step in (5, 10, 15, 20):
                    bs, ewc, est, _ = stored_signals(S, th_, b)
                    log.append(dict(bracket_stored=bs, ewc_penalty=ewc, est_watch=est,
                                    bracket_full=float(bracket(th_, a, b).norm()),
                                    simple_check=evaluate(th_, a, "est")[1] - base_tr[a]))
            thAB = train(th_task[a], b, seed, record=rec)
            gB0 = grad(th_task[a], b)
            accA1, lossA1 = evaluate(thAB, a)
            row = dict(seed=seed, A=a, B=b, lr=lr, acc_A_before=base[a][0], acc_A_after=accA1,
                       forgetting=base[a][0] - accA1, loss_increase=lossA1 - base[a][1],
                       curv_stored_switch=float(gB0 @ H_stored(S, gB0)))
            for k in ["bracket_stored", "ewc_penalty", "est_watch", "bracket_full", "simple_check"]:
                row[k] = float(np.mean([x[k] for x in log]))
            rows.append(row)
        print(f"  seed {seed} done after {time.time() - t0:.0f}s | mean task accuracy {np.mean([base[t][0] for t in range(5)]):.4f}")
    with open(os.path.join(out, "results.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    col = lambda k: np.array([r[k] for r in rows])
    Sm = {"scenario": scenario, "lr": lr, "seeds": seeds, "n_pairs": len(rows), "K": K,
          "mean_task_accuracy": float(col("acc_A_before").mean()),
          "forgetting_mean": float(col("forgetting").mean()), "forgetting_max": float(col("forgetting").max()), "rho": {}}
    for k in SIGNALS:
        r_, p_ = spearmanr(col(k), col("loss_increase")); Sm["rho"][k] = {"rho": float(r_), "p": float(p_)}
    def resid(yv, x):
        x1 = np.c_[np.ones_like(x), x]; return yv - x1 @ np.linalg.lstsq(x1, yv, rcond=None)[0]
    rl = rankdata(col("loss_increase"))
    for c in ["ewc_penalty", "est_watch"]:
        rk, rc = rankdata(col("bracket_stored")), rankdata(col(c))
        Sm[f"partial_bracket_stored_given_{c}"] = float(np.corrcoef(resid(rk, rc), resid(rl, rc))[0, 1])
    json.dump(Sm, open(os.path.join(out, "summary.json"), "w"), indent=2)
    print(f"finished {tag} in {time.time() - t0:.0f}s")
    return Sm, rows

def show(Sm):
    print(f"\n== {Sm['scenario']} | lr {Sm['lr']} | {Sm['seeds']} seeds | task acc {Sm['mean_task_accuracy']:.3f} | "
          f"forgetting mean {Sm['forgetting_mean']:.3f}, max {Sm['forgetting_max']:.3f}")
    for k, name in SIGNALS.items():
        print(f"  {name:52s} {Sm['rho'][k]['rho']:6.2f}")
    print(f"  extra info from stored bracket beyond EWC penalty:        {Sm['partial_bracket_stored_given_ewc_penalty']:6.2f}")
    print(f"  extra info from stored bracket beyond estimated watching: {Sm['partial_bracket_stored_given_est_watch']:6.2f}")
print("defined")

## Cell 8: main run, learning rate 0.05 (same as before), both settings
Prints a line per seed, then the table for each setting. **Send Claude the two tables.**

In [ ]:
RES = {}
for sc in ["task", "domain"]:
    RES[(sc, 0.05)] = run(sc, lr=0.05)
    show(RES[(sc, 0.05)][0])

## Cell 9: learning-rate sweep (robustness)
Same test at a smaller and a larger learning rate. Takes about twice as long as Cell 8.
If time is short, run only one of them by editing the list.

In [ ]:
for lr in [0.02, 0.1]:
    for sc in ["task", "domain"]:
        RES[(sc, lr)] = run(sc, lr=lr)
        show(RES[(sc, lr)][0])

In [6]:
LR = 0.05; setup("domain")
pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
STEPS = (5, 10, 15, 20)
per = {s: {k: [] for k in ["bracket_stored", "ewc_penalty", "est_watch"]} for s in STEPS}
target = []
t0 = time.time()
for seed in range(SEEDS):
    th0 = init(seed)
    th_task = {t: train(th0, t, seed) for t in range(5)}
    base = {t: evaluate(th_task[t], t) for t in range(5)}
    summ = {t: make_summary(th_task[t], t) for t in range(5)}
    for a, b in pairs:
        S = summ[a]; log = {}
        def rec(step, th_, S=S, b=b, log=log):
            if step in STEPS:
                bs, ewc, est, _ = stored_signals(S, th_, b)
                log[step] = (bs, ewc, est)
        thAB = train(th_task[a], b, seed, record=rec)
        target.append(evaluate(thAB, a)[1] - base[a][1])
        for s in STEPS:
            for k, v in zip(["bracket_stored", "ewc_penalty", "est_watch"], log[s]):
                per[s][k].append(v)
    print(f"seed {seed} done after {time.time() - t0:.0f}s")
print(f"\n{'step':>5} {'bracket(stored)':>16} {'EWC penalty':>12} {'est. watching':>14}")
for s in STEPS:
    r = [spearmanr(per[s][k], target)[0] for k in ["bracket_stored", "ewc_penalty", "est_watch"]]
    print(f"{s:>5} {r[0]:16.2f} {r[1]:12.2f} {r[2]:14.2f}")

seed 0 done after 49s
seed 1 done after 96s
seed 2 done after 144s
seed 3 done after 190s
seed 4 done after 238s

 step  bracket(stored)  EWC penalty  est. watching
    5             0.45         0.59           0.38
   10             0.38         0.60           0.41
   15             0.36         0.59           0.42
   20             0.37         0.59           0.40


## Cell 10: one summary table and one chart
Saves `stored_summary.png` in the results folder.

In [7]:
def grad_on(th, X, y, cls):
    th = th.detach().requires_grad_(True)
    return torch.autograd.grad(task_loss(th, X, y, cls), th)[0].detach()

def hvp_on(th, X, y, cls, v):
    th = th.detach().requires_grad_(True)
    g = torch.autograd.grad(task_loss(th, X, y, cls), th, create_graph=True)[0]
    return torch.autograd.grad(g @ v.detach(), th)[0].detach()

@torch.no_grad()
def loss_on(th, X, y, cls):
    return float(task_loss(th, X, y, cls))

BUF = (10, 50, 200)

def run_buffer(scenario, lr=0.05, seeds=SEEDS):
    global LR
    LR = lr; setup(scenario)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    keys = ["bracket_stored", "ewc_penalty"] + [f"bracket_buf{n}" for n in BUF] + [f"check_buf{n}" for n in BUF]
    vals = {k: [] for k in keys}; target = []; t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base = {t: evaluate(th_task[t], t) for t in range(5)}
        summ = {t: make_summary(th_task[t], t) for t in range(5)}
        buf = {}
        for t in range(5):  # tiny buffer: n random training images of task t, kept after training t
            d = data[t]; g = torch.Generator().manual_seed(seed * 77 + t)
            idx = torch.randperm(len(d["ytr"]), generator=g)[:max(BUF)].to(dev)
            buf[t] = {n: (d["Xtr"][idx[:n]], d["ytr"][idx[:n]]) for n in BUF}
        for a, b in pairs:
            S = summ[a]; cA = data[a]["cls"]; log = []
            base_buf = {n: loss_on(th_task[a], *buf[a][n], cA) for n in BUF}
            def rec(step, th_, S=S, a=a, b=b, cA=cA, log=log, base_buf=base_buf):
                if step in (5, 10, 15, 20):
                    bs, ewc, est, gB = stored_signals(S, th_, b)
                    row = {"bracket_stored": bs, "ewc_penalty": ewc}
                    for n in BUF:
                        Xb, yb = buf[a][n]
                        gA = grad_on(th_, Xb, yb, cA)
                        row[f"bracket_buf{n}"] = float((hvp(th_, b, gA) - hvp_on(th_, Xb, yb, cA, gB)).norm())
                        row[f"check_buf{n}"] = loss_on(th_, Xb, yb, cA) - base_buf[n]
                    log.append(row)
            thAB = train(th_task[a], b, seed, record=rec)
            target.append(evaluate(thAB, a)[1] - base[a][1])
            for k in keys:
                vals[k].append(float(np.mean([r[k] for r in log])))
        print(f"  seed {seed} done after {time.time() - t0:.0f}s")
    rho = {k: spearmanr(vals[k], target)[0] for k in keys}
    print(f"\n== {scenario} | lr {lr}")
    print(f"  {'no old data':14s} bracket (stored) {rho['bracket_stored']:.2f} | EWC penalty {rho['ewc_penalty']:.2f}")
    for n in BUF:
        print(f"  buffer {n:>4} imgs  bracket {rho[f'bracket_buf{n}']:.2f} | simple check {rho[f'check_buf{n}']:.2f}")
    json.dump({"scenario": scenario, "lr": lr, "rho": rho},
              open(os.path.join(RESULTS_DIR, f"buffer_{scenario}_lr{lr}.json"), "w"), indent=2)
    return rho

for sc in ["domain", "task"]:
    run_buffer(sc)

  seed 0 done after 51s
  seed 1 done after 103s
  seed 2 done after 156s
  seed 3 done after 206s
  seed 4 done after 256s

== domain | lr 0.05
  no old data    bracket (stored) 0.40 | EWC penalty 0.59
  buffer   10 imgs  bracket 0.72 | simple check 0.71
  buffer   50 imgs  bracket 0.72 | simple check 0.79
  buffer  200 imgs  bracket 0.72 | simple check 0.81
  seed 0 done after 58s
  seed 1 done after 118s
  seed 2 done after 176s
  seed 3 done after 232s
  seed 4 done after 288s

== task | lr 0.05
  no old data    bracket (stored) 0.81 | EWC penalty 0.58
  buffer   10 imgs  bracket 0.66 | simple check 0.12
  buffer   50 imgs  bracket 0.71 | simple check 0.31
  buffer  200 imgs  bracket 0.77 | simple check 0.42


In [8]:
for lr in [0.02, 0.1]:
    for sc in ["domain", "task"]:
        run_buffer(sc, lr=lr)

  seed 0 done after 49s
  seed 1 done after 101s
  seed 2 done after 153s
  seed 3 done after 205s
  seed 4 done after 256s

== domain | lr 0.02
  no old data    bracket (stored) 0.46 | EWC penalty 0.61
  buffer   10 imgs  bracket 0.74 | simple check 0.68
  buffer   50 imgs  bracket 0.75 | simple check 0.71
  buffer  200 imgs  bracket 0.77 | simple check 0.76
  seed 0 done after 59s
  seed 1 done after 117s
  seed 2 done after 174s
  seed 3 done after 229s
  seed 4 done after 284s

== task | lr 0.02
  no old data    bracket (stored) 0.86 | EWC penalty 0.55
  buffer   10 imgs  bracket 0.73 | simple check 0.01
  buffer   50 imgs  bracket 0.79 | simple check 0.14
  buffer  200 imgs  bracket 0.83 | simple check 0.24
  seed 0 done after 51s
  seed 1 done after 103s
  seed 2 done after 155s
  seed 3 done after 205s
  seed 4 done after 255s

== domain | lr 0.1
  no old data    bracket (stored) 0.33 | EWC penalty 0.49
  buffer   10 imgs  bracket 0.57 | simple check 0.72
  buffer   50 imgs  bra

In [9]:
import glob
RES = {}
for f in glob.glob(os.path.join(RESULTS_DIR, "*", "summary.json")):
    Sm = json.load(open(f))
    RES[(Sm["scenario"], Sm["lr"])] = (Sm, None)
print("loaded", sorted(RES))

loaded [('domain', 0.02), ('domain', 0.05), ('domain', 0.1), ('task', 0.02), ('task', 0.05), ('task', 0.1)]


In [10]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Calibri", "font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
keys = ["bracket_stored", "ewc_penalty", "est_watch", "curv_stored_switch", "bracket_full", "simple_check"]
short = ["Bracket (stored)", "EWC penalty", "Est. watching", "EWC curvature", "Ref: full bracket", "Ref: simple check"]
runs = sorted(RES)
print(f"{'setting':16s}" + "".join(f"{s:>18s}" for s in short))
for r_ in runs:
    Sm = RES[r_][0]
    print(f"{r_[0] + ' lr ' + str(r_[1]):16s}" + "".join(f"{Sm['rho'][k]['rho']:18.2f}" for k in keys))
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), dpi=200, sharey=True)
cols = ["#1F8A70", "#B8651C", "#4A6FA5", "#9AA6A3", "#C9D3D0", "#E3E8E6"]
for ax, sc in zip(axs, ["task", "domain"]):
    lrs = sorted(lr for s, lr in RES if s == sc); x = np.arange(len(lrs)); w = 0.13
    for j, k in enumerate(keys):
        ax.bar(x + (j - 2.5) * w, [RES[(sc, lr)][0]["rho"][k]["rho"] for lr in lrs], w, color=cols[j], label=short[j])
    ax.set_xticks(x); ax.set_xticklabels([f"lr {lr}" for lr in lrs]); ax.axhline(0, color="#9AA6A3", lw=0.8)
    ax.set_title(("Mild" if sc == "task" else "Strong") + " forgetting", loc="left", weight="bold")
    ax.grid(axis="y", color="#E3E8E6"); ax.set_axisbelow(True)
axs[0].set_ylabel("Rank correlation with forgetting")
axs[1].legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Predicting forgetting without the old data (MNIST)", x=0.01, ha="left", weight="bold")
fig.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "stored_summary.png"), facecolor="white", bbox_inches="tight"); plt.close(fig)
print("saved", os.path.join(RESULTS_DIR, "stored_summary.png"))

setting           Bracket (stored)       EWC penalty     Est. watching     EWC curvature Ref: full bracket Ref: simple check
domain lr 0.02                0.46              0.61              0.46              0.28              0.72              0.77
domain lr 0.05                0.40              0.59              0.41              0.25              0.69              0.81
domain lr 0.1                 0.33              0.49              0.37              0.28              0.63              0.82
task lr 0.02                  0.86              0.55              0.58              0.42              0.86              0.61
task lr 0.05                  0.81              0.58              0.67              0.42              0.82              0.81
task lr 0.1                   0.71              0.59              0.79              0.40              0.73              0.84
saved F:\forgetting\results_stored\stored_summary.png


In [11]:
import glob
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Calibri", "font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
GREEN, AMBER, BLUE, GRAY, GRID = "#1F8A70", "#B8651C", "#4A6FA5", "#9AA6A3", "#E3E8E6"

# ---- load buffer results ----
BUFRES = {}
for f in glob.glob(os.path.join(RESULTS_DIR, "buffer_*.json")):
    d = json.load(open(f)); BUFRES[(d["scenario"], d["lr"])] = d["rho"]
LRS = sorted({lr for _, lr in BUFRES})
print("loaded", sorted(BUFRES))

# ---- summary table + CSV ----
cols = ["ewc_penalty", "bracket_stored", "bracket_buf10", "check_buf10", "bracket_buf50", "check_buf50", "bracket_buf200", "check_buf200"]
names = ["EWC (no data)", "Bracket (no data)", "Bracket 10", "Check 10", "Bracket 50", "Check 50", "Bracket 200", "Check 200"]
print(f"\n{'setting':20s}" + "".join(f"{n:>18s}" for n in names))
rows = []
for sc in ["task", "domain"]:
    for lr in LRS:
        r = BUFRES[(sc, lr)]
        label = ("slow" if sc == "task" else "fast") + f" lr {lr}"
        print(f"{label:20s}" + "".join(f"{r[c]:18.2f}" for c in cols))
        rows.append({"setting": label, **{n: round(r[c], 3) for n, c in zip(names, cols)}})
with open(os.path.join(RESULTS_DIR, "buffer_summary.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

# ---- chart 1: buffer sizes ----
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), dpi=200, sharey=True)
for ax, sc in zip(axs, ["task", "domain"]):
    x = np.arange(len(LRS)); w = 0.11
    series = [("EWC (no data)", "ewc_penalty", AMBER), ("Bracket 10", "bracket_buf10", "#7CC4AE"),
              ("Bracket 50", "bracket_buf50", "#3FA585"), ("Bracket 200", "bracket_buf200", GREEN),
              ("Check 10", "check_buf10", "#C7D2E6"), ("Check 50", "check_buf50", "#8EA6CC"), ("Check 200", "check_buf200", BLUE)]
    for j, (lab, k, c) in enumerate(series):
        ax.bar(x + (j - 3) * w, [BUFRES[(sc, lr)][k] for lr in LRS], w, color=c, label=lab)
    ax.set_xticks(x); ax.set_xticklabels([f"lr {lr}" for lr in LRS]); ax.axhline(0, color=GRAY, lw=0.8)
    ax.set_title(("Slow" if sc == "task" else "Fast") + " forgetting", loc="left", weight="bold")
    ax.grid(axis="y", color=GRID); ax.set_axisbelow(True)
axs[0].set_ylabel("Rank correlation with forgetting")
axs[1].legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Bracket vs simple check with a tiny buffer of old images (MNIST)", x=0.01, ha="left", weight="bold")
fig.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "buffer_summary.png"), facecolor="white", bbox_inches="tight"); plt.close(fig)

# ---- chart 2: headline, bracket vs EWC everywhere ----
fig, ax = plt.subplots(figsize=(9, 4.6), dpi=200)
labels, br, ew = [], [], []
for sc, key, tag in [("task", "bracket_stored", "Slow, no data"), ("domain", "bracket_buf10", "Fast, 10 images")]:
    for lr in LRS:
        labels.append(f"{tag}\nlr {lr}"); br.append(BUFRES[(sc, lr)][key]); ew.append(BUFRES[(sc, lr)]["ewc_penalty"])
x = np.arange(len(labels)); w = 0.38
ax.bar(x - w / 2, br, w, color=GREEN, label="Lie bracket")
ax.bar(x + w / 2, ew, w, color=AMBER, label="EWC (no data)")
for xi, v in zip(x - w / 2, br): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=10)
for xi, v in zip(x + w / 2, ew): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10); ax.set_ylim(0, 1)
ax.set_ylabel("Rank correlation with forgetting"); ax.grid(axis="y", color=GRID); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper right")
ax.set_title("The Lie bracket predicts forgetting better than EWC in every setting tested (MNIST)", loc="left", weight="bold", fontsize=12)
fig.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "headline_bracket_vs_ewc.png"), facecolor="white"); plt.close(fig)
print("\nsaved buffer_summary.csv, buffer_summary.png, headline_bracket_vs_ewc.png in", RESULTS_DIR)

loaded [('domain', 0.02), ('domain', 0.05), ('domain', 0.1), ('task', 0.02), ('task', 0.05), ('task', 0.1)]

setting                  EWC (no data) Bracket (no data)        Bracket 10          Check 10        Bracket 50          Check 50       Bracket 200         Check 200
slow lr 0.02                      0.55              0.86              0.73              0.01              0.79              0.14              0.83              0.24
slow lr 0.05                      0.58              0.81              0.66              0.12              0.71              0.31              0.77              0.42
slow lr 0.1                       0.59              0.71              0.57              0.11              0.65              0.32              0.69              0.59
fast lr 0.02                      0.61              0.46              0.74              0.68              0.75              0.71              0.77              0.76
fast lr 0.05                      0.59              0.40          

In [12]:
from torchvision import datasets
datasets.CIFAR10(DATA_DIR, train=True, download=True)
datasets.CIFAR10(DATA_DIR, train=False, download=True)
print("CIFAR-10 ready in", DATA_DIR)

100%|██████████| 170M/170M [13:47<00:00, 206kB/s]    


Extracting F:\forgetting\data\cifar-10-python.tar.gz to F:\forgetting\data
Files already downloaded and verified
CIFAR-10 ready in F:\forgetting\data


In [13]:
DS = "cifar" if "cifar" in RESULTS_DIR else "mnist"

def run_order(scenario, lr=0.05, seeds=3):
    global LR
    LR = lr; setup(scenario)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    K_ = ["br_start", "br_onA_start", "br_early", "forgetting_loss"]
    vals = {k: [] for k in K_}; gapA, gapP = [], []; t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base = {t: evaluate(th_task[t], t)[1] for t in range(5)}
        g0 = {t: grad(th0, t) for t in range(5)}
        thAB, early = {}, {}
        for a, b in pairs:
            log = []
            def rec(step, th_, a=a, b=b, log=log):
                if step in (5, 10, 15, 20):
                    log.append(float((hvp(th_, b, grad(th_, a)) - hvp(th_, a, grad(th_, b))).norm()))
            thAB[(a, b)] = train(th_task[a], b, seed, record=rec)
            early[(a, b)] = float(np.mean(log))
        for a, b in pairs:
            br0 = hvp(th0, b, g0[a]) - hvp(th0, a, g0[b])
            lossA_AB = evaluate(thAB[(a, b)], a)[1]          # A learned first, then B
            lossA_BA = evaluate(thAB[(b, a)], a)[1]          # A learned last
            gapA.append(lossA_AB - lossA_BA)
            gapP.append(float((thAB[(a, b)] - thAB[(b, a)]).norm()))
            vals["br_start"].append(float(br0.norm()))
            vals["br_onA_start"].append(float(g0[a] @ br0))
            vals["br_early"].append(early[(a, b)])
            vals["forgetting_loss"].append(lossA_AB - base[a])
        print(f"  seed {seed} done after {time.time() - t0:.0f}s")
    r = lambda x, y: float(spearmanr(x, y)[0])
    out = {
        "order_gap_on_A vs br_start": r(vals["br_start"], gapA),
        "order_gap_on_A vs br_onA_start": r(vals["br_onA_start"], gapA),
        "order_gap_on_A vs br_early": r(vals["br_early"], gapA),
        "weight_gap vs br_start": r(vals["br_start"], gapP),
        "weight_gap vs br_early": r(vals["br_early"], gapP),
        "forgetting vs order_gap_on_A": r(gapA, vals["forgetting_loss"]),
        "forgetting vs br_early": r(vals["br_early"], vals["forgetting_loss"]),
    }
    share = float(np.mean(np.abs(gapA)) / np.mean(np.abs(vals["forgetting_loss"])))
    Sm = {"dataset": DS, "scenario": scenario, "lr": lr, "seeds": seeds, "rho": out, "order_share_of_forgetting": share}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"order_{DS}_{scenario}_lr{lr}.json"), "w"), indent=2)
    print(f"\n== ORDER | {DS} | {'slow' if scenario == 'task' else 'fast'} | lr {lr}")
    for k, v in out.items(): print(f"  {k:34s} {v:6.2f}")
    print(f"  order effect as share of forgetting  {share:6.2f}")
    return Sm

for sc in ["task", "domain"]:
    run_order(sc, lr=0.05)

  seed 0 done after 38s
  seed 1 done after 76s
  seed 2 done after 114s

== ORDER | mnist | slow | lr 0.05
  order_gap_on_A vs br_start           0.33
  order_gap_on_A vs br_onA_start      -0.09
  order_gap_on_A vs br_early           0.83
  weight_gap vs br_start               0.64
  weight_gap vs br_early               0.66
  forgetting vs order_gap_on_A         1.00
  forgetting vs br_early               0.82
  order effect as share of forgetting    1.04
  seed 0 done after 34s
  seed 1 done after 66s
  seed 2 done after 100s

== ORDER | mnist | fast | lr 0.05
  order_gap_on_A vs br_start          -0.03
  order_gap_on_A vs br_onA_start       0.65
  order_gap_on_A vs br_early           0.69
  weight_gap vs br_start              -0.24
  weight_gap vs br_early               0.69
  forgetting vs order_gap_on_A         1.00
  forgetting vs br_early               0.69
  order effect as share of forgetting    1.00


In [5]:
DS = "cifar" if "cifar" in RESULTS_DIR else "mnist"

def run_mech(scenario, lr=0.05, seeds=3):
    global LR
    LR = lr; setup(scenario)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    R = {k: [] for k in ["forget", "push_ratio", "gA_norm", "term_HBgA", "term_HAgB", "bracket",
                         "slope20", "curv20", "actual20", "slope_end", "curv_end", "actual_end"]}
    t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base_te = {t: evaluate(th_task[t], t)[1] for t in range(5)}
        base_est = {t: evaluate(th_task[t], t, "est")[1] for t in range(5)}
        gA0 = {t: grad(th_task[t], t) for t in range(5)}          # A's push when A finishes
        for a, b in pairs:
            thA = th_task[a]; log = []; at20 = {}
            def rec(step, th_, a=a, b=b, thA=thA, log=log, at20=at20):
                if step not in (5, 10, 15, 20): return
                gA, gB = grad(th_, a), grad(th_, b)
                t1, t2 = hvp(th_, b, gA), hvp(th_, a, gB)
                log.append((float(t1.norm()), float(t2.norm()), float((t1 - t2).norm())))
                if step == 20: at20["th"] = th_.detach().clone()
            thAB = train(thA, b, seed, record=rec)
            def split(th_):
                d = th_ - thA
                slope = float(gA0[a] @ d)
                curv = float(0.5 * d @ hvp(thA, a, d))
                actual = evaluate(th_, a, "est")[1] - base_est[a]
                return slope, curv, actual
            s20, c20, a20 = split(at20["th"]); se, ce, ae = split(thAB)
            R["forget"].append(evaluate(thAB, a)[1] - base_te[a])
            R["gA_norm"].append(float(gA0[a].norm()))
            R["push_ratio"].append(float(gA0[a].norm() / grad(thA, b).norm()))
            R["term_HBgA"].append(float(np.mean([x[0] for x in log])))
            R["term_HAgB"].append(float(np.mean([x[1] for x in log])))
            R["bracket"].append(float(np.mean([x[2] for x in log])))
            for k, v in zip(["slope20", "curv20", "actual20", "slope_end", "curv_end", "actual_end"], [s20, c20, a20, se, ce, ae]):
                R[k].append(v)
        print(f"  seed {seed} done after {time.time() - t0:.0f}s")
    r = lambda k: float(spearmanr(R[k], R["forget"])[0])
    share = lambda s, c: float(np.mean(np.abs(R[s])) / (np.mean(np.abs(R[s])) + np.mean(np.abs(R[c]))))
    Sm = {"dataset": DS, "scenario": scenario, "lr": lr, "seeds": seeds,
          "c1_push_A_mean": float(np.mean(R["gA_norm"])), "c1_push_ratio_A_over_B": float(np.mean(R["push_ratio"])),
          "c2_slope_share_step20": share("slope20", "curv20"), "c2_slope_share_end": share("slope_end", "curv_end"),
          "c2_taylor_vs_actual_step20": float(spearmanr(np.array(R["slope20"]) + np.array(R["curv20"]), R["actual20"])[0]),
          "c3_rho_full_bracket": r("bracket"), "c3_rho_HBgA": r("term_HBgA"), "c3_rho_HAgB": r("term_HAgB"),
          "c3_rho_slope20": r("slope20"), "c3_rho_curv20": r("curv20")}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"mech_{DS}_{scenario}_lr{lr}.json"), "w"), indent=2)
    print(f"\n== MECH | {DS} | {'slow' if scenario == 'task' else 'fast'} | lr {lr}")
    for k, v in Sm.items():
        if k.startswith("c"): print(f"  {k:30s} {v:8.3f}")
    return Sm
print("defined")

defined


In [6]:
for sc in ["task", "domain"]:
    run_mech(sc, lr=0.05)

  seed 0 done after 39s
  seed 1 done after 75s
  seed 2 done after 113s

== MECH | mnist | slow | lr 0.05
  c1_push_A_mean                    0.148
  c1_push_ratio_A_over_B            0.072
  c2_slope_share_step20             0.313
  c2_slope_share_end                0.178
  c2_taylor_vs_actual_step20        0.976
  c3_rho_full_bracket               0.823
  c3_rho_HBgA                       0.754
  c3_rho_HAgB                       0.780
  c3_rho_slope20                    0.011
  c3_rho_curv20                     0.840
  seed 0 done after 33s
  seed 1 done after 65s
  seed 2 done after 97s

== MECH | mnist | fast | lr 0.05
  c1_push_A_mean                    0.150
  c1_push_ratio_A_over_B            0.029
  c2_slope_share_step20             0.106
  c2_slope_share_end                0.084
  c2_taylor_vs_actual_step20        0.602
  c3_rho_full_bracket               0.694
  c3_rho_HBgA                       0.684
  c3_rho_HAgB                       0.790
  c3_rho_slope20               

In [7]:
import glob
rows = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(RESULTS_DIR, "mech_*.json")))]
keys = ["c1_push_A_mean", "c1_push_ratio_A_over_B", "c2_slope_share_step20", "c2_slope_share_end",
        "c2_taylor_vs_actual_step20", "c3_rho_full_bracket", "c3_rho_HAgB", "c3_rho_HBgA", "c3_rho_slope20", "c3_rho_curv20"]
print(f"{'':30s}" + "".join(f"{r['dataset'] + ' ' + ('slow' if r['scenario'] == 'task' else 'fast'):>14s}" for r in rows))
for k in keys:
    print(f"{k:30s}" + "".join(f"{r[k]:14.3f}" for r in rows))

                                  mnist fast    mnist slow
c1_push_A_mean                         0.150         0.148
c1_push_ratio_A_over_B                 0.029         0.072
c2_slope_share_step20                  0.106         0.313
c2_slope_share_end                     0.084         0.178
c2_taylor_vs_actual_step20             0.602         0.976
c3_rho_full_bracket                    0.694         0.823
c3_rho_HAgB                            0.790         0.780
c3_rho_HBgA                            0.684         0.754
c3_rho_slope20                         0.370         0.011
c3_rho_curv20                          0.468         0.840


In [5]:
from scipy.sparse.linalg import LinearOperator, eigsh
if "DS" not in globals(): DS = "cifar" if "cifar" in RESULTS_DIR else "mnist"

def curv_shape(scenario, lr=0.05, seeds=2):
    global LR
    LR = lr; setup(scenario)
    lmax, lmin, cosv = [], [], []; t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        for a in range(5):
            th = th_task[a]
            def mv(v, th=th, a=a):
                v_t = torch.as_tensor(np.asarray(v).ravel(), dtype=torch.float32, device=dev)
                return hvp(th, a, v_t).double().cpu().numpy()
            op = LinearOperator((NPARAM, NPARAM), matvec=mv, dtype=np.float64)
            lmax.append(float(eigsh(op, k=1, which="LA", tol=1e-3, return_eigenvectors=False).max()))
            lmin.append(float(eigsh(op, k=1, which="SA", tol=1e-3, return_eigenvectors=False).min()))
            for b in range(5):
                if b != a:
                    gB = grad(th, b); cosv.append(float(F.cosine_similarity(gB, hvp(th, a, gB), dim=0)))
        print(f"  seed {seed} done after {time.time() - t0:.0f}s")
    ratio = [abs(min(mn, 0)) / mx for mn, mx in zip(lmin, lmax)]
    Sm = {"dataset": DS, "scenario": scenario, "lr": lr, "seeds": seeds,
          "largest_curvature_mean": float(np.mean(lmax)), "most_negative_mean": float(np.mean(lmin)),
          "neg_to_pos_ratio": float(np.mean(ratio)), "cos_gB_HAgB_mean": float(np.mean(cosv)),
          "cos_below_0.3_share": float(np.mean(np.array(cosv) < 0.3))}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"curv_{DS}_{scenario}_lr{lr}.json"), "w"), indent=2)
    print(f"\n== VALLEY | {DS} | {'slow' if scenario == 'task' else 'fast'} | lr {lr}")
    for k, v in Sm.items():
        if k not in ("dataset", "scenario", "lr", "seeds"): print(f"  {k:24s} {v:9.4f}")
    return Sm

for sc in ["task", "domain"]:
    curv_shape(sc, lr=0.05)

  seed 0 done after 14s
  seed 1 done after 28s

== VALLEY | mnist | slow | lr 0.05
  largest_curvature_mean      5.4199
  most_negative_mean         -0.2883
  neg_to_pos_ratio            0.0629
  cos_gB_HAgB_mean            0.0673
  cos_below_0.3_share         1.0000
  seed 0 done after 12s
  seed 1 done after 24s

== VALLEY | mnist | fast | lr 0.05
  largest_curvature_mean      5.4304
  most_negative_mean         -0.3038
  neg_to_pos_ratio            0.0647
  cos_gB_HAgB_mean            0.5885
  cos_below_0.3_share         0.0000
